> ¿Qué hago cuando sé que algo debe existir, pero todavía no sé exactamente cómo se implementará?

# Fase 4 — Abstracción y Contratos

# 10. Clases Abstractas



## Problema

Supongamos que nuestro sistema tiene distintos tipos de tickets:

```text
Ticket de Soporte
Ticket de Desarrollo
Ticket de Infraestructura
```
Todos tienen algo en común:

```text
Abrir
Cerrar
Mostrar información
```

Pero cada uno puede comportarse diferente.


## Solución básicas

```python
class Ticket:
    pass
```

y luego cada programador implementa lo que quiera.

Problema:

```text
No hay reglas.
No hay contrato.
No hay garantía.
```

## Solución profesional

Crear una clase abstracta.

Una clase abstracta representa una idea.

No representa un objeto real.

In [ ]:
from abc import ABC

class TicketBase(ABC):
    pass


In [ ]:
from abc import ABC, abstractmethod

class TicketBase(ABC):

    @abstractmethod
    def atender(self):
        pass


Toda clase hija deberá implementar:

```python
def atender(self):
```

In [ ]:
class TicketSoporte:
    def entender(self):
        print("Entendiendo el ticket de soporte")

In [ ]:
class TicketSoporte(TicketBase):

    def atender(self):
        print("Atendiendo ticket de soporte")

In [ ]:
class TicketDesarrollo(TicketBase):
    def atender(self):
        print("Atendiendo ticket de desarrollo")

## Contrato

La clase abstracta funciona como un contrato.

> Si heredas de mí, debes implementar estos métodos.


# 12. @classmethod

Ahora aparece otro decorador.

In [ ]:
# sin decorador y mala implementación
class Ticket:

    contador = 0

    def __init__(self):
        Ticket.contador += 1

    
    def total_tickets(self):
        return self.contador

In [ ]:

t1 = Ticket()
t2 = Ticket()
t3 = Ticket()

Ticket.total_tickets()

In [ ]:
class Ticket:

    contador = 0

    def __init__(self):
        Ticket.contador += 1

    @classmethod
    def total_tickets(cls):
        return cls.contador

In [ ]:
t1 = Ticket()
t2 = Ticket()
t3 = Ticket()

print(Ticket.total_tickets())

No existe un objeto.

Trabaja directamente con la clase.

### ¿Para qué sirve?

Cuando la información pertenece a la clase completa.

Ejemplo:

```text
Cantidad total de tickets
Configuración global
Factores compartidos
```


### 13. @staticmethod

Ahora otro decorador.

#### Problema

A veces una función está relacionada con la clase.

Pero no necesita:

```text
self
ni cls
```

Ejemplo: Validar un folio.

In [ ]:
import re

class Ticket:    
    @staticmethod
    def es_folio_valido(folio):
        return bool(re.fullmatch(r"^TK-?\d{3}$", folio))

In [ ]:
Ticket.es_folio_valido("TK453")

## Mapa completo hasta ahora

```text
POO
│
├── Clase
├── Constructor
├── Atributos
├── Métodos
├── __str__
├── Docstrings
│
├── Encapsulación
├── @property
├── Decoradores básicos
│
├── Diseño
│   ├── Detectar patrones
│   └── Identificar entidades
│
└── Abstracción
    ├── Clase abstracta
    ├── Método abstracto
    ├── @classmethod
    ├── @staticmethod
    └── self vs cls

In [ ]:
from abc import ABC, abstractmethod
import re


class TicketBase(ABC):
    ESTADOS_VALIDOS = ["Abierto", "En proceso", "Cerrado"]
    _contador_global = 0

    def __init__(self, folio, titulo, descripcion):
        self.folio = folio
        self.titulo = titulo
        self.descripcion = descripcion
        self.estado = "Abierto"

        TicketBase._contador_global += 1

    @property
    def folio(self):
        return self._folio

    @folio.setter
    def folio(self, nuevo_folio):
        if not self.es_folio_valido(nuevo_folio):
            raise ValueError(f"Folio no válido: {nuevo_folio}")

        self._folio = nuevo_folio

    @property
    def titulo(self):
        return self._titulo

    @titulo.setter
    def titulo(self, nuevo_titulo):
        self._titulo = nuevo_titulo

    @property
    def descripcion(self):
        return self._descripcion

    @descripcion.setter
    def descripcion(self, nueva_descripcion):
        self._descripcion = nueva_descripcion

    @property
    def estado(self):
        return self._estado

    @estado.setter
    def estado(self, nuevo_estado):
        if nuevo_estado not in self.ESTADOS_VALIDOS:
            raise ValueError(f"Estado no válido: {nuevo_estado}")

        self._estado = nuevo_estado

    def cambiar_estado(self, nuevo_estado):
        self.estado = nuevo_estado

    def cerrar(self):
        self.estado = "Cerrado"
        # invoar la función que registra en bitácora

    @classmethod
    def total_tickets_creados(cls):
        return cls._contador_global

    @staticmethod
    def es_folio_valido(folio):
        return bool(re.fullmatch(r"^TK-?\d{3}$", folio))

    @abstractmethod
    def atender(self):
        pass

    def __str__(self):
        return (
            f"{self.__class__.__name__} | "
            f"Folio: {self.folio} | "
            f"Título: {self.titulo} | "
            f"Estado: {self.estado}"
        )


class TicketSoporte(TicketBase):

    def atender(self):
        self.estado = "En proceso"
        return f"El ticket de soporte {self.folio} está siendo atendido por mesa de ayuda."


class TicketDesarrollo(TicketBase):

    def atender(self):
        self.estado = "En proceso"
        return f"El ticket de desarrollo {self.folio} fue asignado al equipo de programación."


class TicketInfraestructura(TicketBase):

    def atender(self):
        self.estado = "En proceso"
        return f"El ticket de infraestructura {self.folio} fue enviado al área de servidores/redes."


def main():
    tickets = [
        TicketSoporte(
            "TK-001",
            "Error en inicio de sesión",
            "El usuario no puede entrar al sistema."
        ),
        TicketDesarrollo(
            "TK002",
            "Reporte no genera PDF",
            "El botón de exportar no responde."
        ),
        TicketInfraestructura(
            "TK-003",
            "Servidor lento",
            "El servidor principal responde con lentitud."
        ),
        TicketSoporte(
            "TK004",
            "Restablecer contraseña",
            "El usuario olvidó su contraseña."
        ),
        TicketDesarrollo(
            "TK-005",
            "Agregar campo al formulario",
            "Se requiere agregar el campo prioridad."
        )
    ]

    print("=== TICKETS CREADOS ===")
    for ticket in tickets:
        print(ticket)

    print("\n=== ATENDER TICKETS ===")
    for ticket in tickets:
        print(ticket.atender())

    print("\n=== ESTADO ACTUALIZADO ===")
    for ticket in tickets:
        print(ticket)

    print("\n=== CERRAR UN TICKET ===")
    tickets[0].cerrar()
    print(tickets[0])

    print("\n=== CONTADOR GLOBAL ===")
    print("Total de tickets creados:", TicketBase.total_tickets_creados())

    print("\n=== VALIDACIÓN DE FOLIOS ===")
    print("TK-123:", TicketBase.es_folio_valido("TK-123"))
    print("TK123:", TicketBase.es_folio_valido("TK123"))
    print("TK-12:", TicketBase.es_folio_valido("TK-12"))
    print("ABC-123:", TicketBase.es_folio_valido("ABC-123"))


if __name__ == "__main__":
    main()